<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/session08-discussion-1.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Session 8: in-class discussion problem (1 of 3)

This is the block 1 question of the lecture quiz. First guess the answer in the quiz, on your own.
Then fix and run the code below, discuss the outcome with your neighbour, and answer the quiz again.

## Regression or classification?

The book page treated the helix fraction of a protein as a **regression** target: predict a number
between 0 and 1. But we could also ask a yes/no question: *is this protein mostly helical*, that is,
is its helix fraction above 0.5? That is a **classification** problem made from the same data.

We have 55 proteins and one feature: the fraction of helix-forming residues (E, M, A, L, K, F, Q).
There are two ways to answer the yes/no question:

- **(a)** train a classifier (logistic regression) directly on the yes/no labels;
- **(b)** train a regression model (linear regression) on the helix fraction itself, and call a
  protein "helical" when its predicted fraction is above 0.5.

**Before you run anything:** which gives the higher accuracy, measured by 5-fold cross-validation?

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import StratifiedKFold

# The lab's 55 proteins from PDBe, with their helix fractions (as in Session 7)
url = "https://raw.githubusercontent.com/ElofssonLab/kb8029-book/main/notebooks/data/session07-helix-dataset.csv"
df = pd.read_csv(url)
helix_frac = (df["helix_residues"] / df["length"]).to_numpy()
HELIX_FORMERS = "EMALKFQ"   # Chou-Fasman helix propensity >= 1.1
former_frac = np.array([sum(s.count(a) for a in HELIX_FORMERS) / len(s) for s in df["sequence"]])

r, p = pearsonr(former_frac, helix_frac)
print(f"{len(df)} proteins; helix fraction {helix_frac.min():.2f}-{helix_frac.max():.2f}")
print(f"Pearson r between helix-former content and helix fraction: {r:.2f} (p = {p:.1e})")

55 proteins; helix fraction 0.03-0.84
Pearson r between helix-former content and helix fraction: 0.71 (p = 1.3e-09)


**Fix and run.** The cell below compares (a) and (b). Run it once as it is and look closely at
the output, in particular how many proteins each approach calls helical. Something is wrong with one
of them: find the line marked `FIX`, change it, and run the cell again. (Hint: scikit-learn's
logistic regression is regularised by default, which keeps its weights small; the parameter `C` sets
how strongly, and a larger `C` means weaker regularisation.)

In [2]:
X = former_frac.reshape(-1, 1)
is_helical = (helix_frac > 0.5).astype(int)
print(f"{is_helical.sum()} of {len(is_helical)} proteins are mostly helical (helix fraction > 0.5)")

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
acc_clf, acc_reg, called_clf, called_reg = [], [], 0, 0
for tr, te in folds.split(X, is_helical):
    clf = LogisticRegression().fit(X[tr], is_helical[tr])    # FIX: why does it never say "helical"?
    pred = clf.predict(X[te])
    acc_clf.append((pred == is_helical[te]).mean()); called_clf += pred.sum()

    reg = LinearRegression().fit(X[tr], helix_frac[tr])      # (b) regress, then threshold
    pred = (reg.predict(X[te]) > 0.5).astype(int)
    acc_reg.append((pred == is_helical[te]).mean()); called_reg += pred.sum()

print(f"(a) logistic regression on the labels: accuracy {np.mean(acc_clf):.3f}, called helical: {called_clf}")
print(f"(b) linear regression, thresholded:    accuracy {np.mean(acc_reg):.3f}, called helical: {called_reg}")
print(f"always answering 'not helical' would score {1 - is_helical.mean():.3f}")

16 of 55 proteins are mostly helical (helix fraction > 0.5)
(a) logistic regression on the labels: accuracy 0.709, called helical: 0
(b) linear regression, thresholded:    accuracy 0.818, called helical: 14
always answering 'not helical' would score 0.709


**Discussion point.** Run as it is, the classifier calls *no* protein helical: its accuracy,
0.709, is exactly what you get by always answering "not helical" (39 of 55). It has learned nothing.
The reason is scikit-learn's default **regularisation** (`C=1`): it keeps the weights small, and our
one feature varies only between about 0.2 and 0.5, so a small weight cannot move the sigmoid past 0.5.
With weaker regularisation, `LogisticRegression(C=100)`, or with the feature standardised first,
the classifier reaches 0.818, the same as the regression.

So which is better? On these 55 proteins, neither: done properly, both draw essentially the same
threshold on one feature. Two lessons:

1. **Compare with the trivial baseline.** An accuracy that equals the majority class is a warning
   sign, however reasonable 0.709 looks.
2. **There is no general winner.** If the continuous measurement is trustworthy, regression keeps
   more information (a protein at 0.49 is not the same as one at 0.05). Direct classification
   optimises the yes/no task itself. Which wins depends on the data, the model and the loss, and one
   comparison on 55 proteins cannot settle it.